# RetailPulse -- Inventory Optimization

**Objective:** Use demand forecasts to calculate optimal reorder points, safety stock levels, and Economic Order Quantity (EOQ) for inventory management.

In [1]:
import os, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (14, 6), "font.size": 12})
FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(FIGURES_DIR, exist_ok=True)
def save_fig(fig, name):
    fig.savefig(os.path.join(FIGURES_DIR, name), dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig); print(f"Saved: {name}")


## Load Forecast and Sales Data

In [2]:
daily = pd.read_csv(os.path.join(PROCESSED_DIR, "daily_sales_features.csv"), parse_dates=["Date"])
forecast = pd.read_csv(os.path.join(PROCESSED_DIR, "prophet_forecast_30d.csv"), parse_dates=["ds"])

print(f"Historical daily sales: {len(daily)} days")
print(f"30-day forecast: {len(forecast)} days")
print(f"Avg daily revenue: {daily['total_revenue'].mean():,.2f}")
print(f"Avg daily quantity: {daily['total_quantity'].mean():,.2f}")


Historical daily sales: 739 days
30-day forecast: 30 days
Avg daily revenue: 23,510.49
Avg daily quantity: 14,226.86


## Demand Statistics

In [3]:
demand_stats = {
    "mean_daily_revenue": daily["total_revenue"].mean(),
    "std_daily_revenue": daily["total_revenue"].std(),
    "mean_daily_quantity": daily["total_quantity"].mean(),
    "std_daily_quantity": daily["total_quantity"].std(),
    "mean_daily_transactions": daily["transaction_count"].mean(),
    "max_daily_revenue": daily["total_revenue"].max(),
    "min_daily_revenue": daily["total_revenue"].min(),
}
stats_df = pd.DataFrame([demand_stats]).T
stats_df.columns = ["Value"]
stats_df["Value"] = stats_df["Value"].apply(lambda x: f"{x:,.2f}")
print("DEMAND STATISTICS")
print("=" * 40)
print(stats_df.to_string())


DEMAND STATISTICS
                              Value
mean_daily_revenue        23,510.49
std_daily_revenue         18,249.05
mean_daily_quantity       14,226.86
std_daily_quantity        12,555.10
mean_daily_transactions       50.03
max_daily_revenue        184,347.66
min_daily_revenue              0.00


## Safety Stock Calculation

Safety stock buffers against demand variability. Formula:

`Safety Stock = Z * sigma_demand * sqrt(lead_time)`

where Z is the service level z-score.

In [4]:
LEAD_TIME_DAYS = 7  # supplier lead time
SERVICE_LEVELS = {"90%": 1.28, "95%": 1.65, "99%": 2.33}

avg_daily_demand = daily["total_quantity"].mean()
std_daily_demand = daily["total_quantity"].std()

safety_stocks = {}
for level, z in SERVICE_LEVELS.items():
    ss = z * std_daily_demand * np.sqrt(LEAD_TIME_DAYS)
    safety_stocks[level] = round(ss)

print("SAFETY STOCK LEVELS")
print("=" * 40)
print(f"Lead time: {LEAD_TIME_DAYS} days")
print(f"Avg daily demand: {avg_daily_demand:,.0f} units")
print(f"Std daily demand: {std_daily_demand:,.0f} units")
print()
for level, ss in safety_stocks.items():
    print(f"  Service Level {level}: {ss:,} units")


SAFETY STOCK LEVELS
Lead time: 7 days
Avg daily demand: 14,227 units
Std daily demand: 12,555 units

  Service Level 90%: 42,519 units
  Service Level 95%: 54,809 units
  Service Level 99%: 77,397 units


## Reorder Point

`Reorder Point = (Avg Daily Demand * Lead Time) + Safety Stock`

In [5]:
reorder_points = {}
for level, ss in safety_stocks.items():
    rop = round(avg_daily_demand * LEAD_TIME_DAYS + ss)
    reorder_points[level] = rop

print("REORDER POINTS")
print("=" * 40)
for level, rop in reorder_points.items():
    print(f"  Service Level {level}: Reorder at {rop:,} units")


REORDER POINTS
  Service Level 90%: Reorder at 142,107 units
  Service Level 95%: Reorder at 154,397 units
  Service Level 99%: Reorder at 176,985 units


## Economic Order Quantity (EOQ)

`EOQ = sqrt(2 * D * S / H)`

where D = annual demand, S = ordering cost, H = holding cost per unit

In [6]:
annual_demand = avg_daily_demand * 365
ordering_cost = 50  # cost per order in GBP
holding_cost_rate = 0.20  # 20% of unit cost per year
avg_unit_cost = daily["total_revenue"].sum() / daily["total_quantity"].sum()
holding_cost = avg_unit_cost * holding_cost_rate

eoq = np.sqrt(2 * annual_demand * ordering_cost / holding_cost)
orders_per_year = annual_demand / eoq
order_cycle = 365 / orders_per_year

print("ECONOMIC ORDER QUANTITY")
print("=" * 40)
print(f"Annual demand: {annual_demand:,.0f} units")
print(f"Ordering cost: {ordering_cost} GBP/order")
print(f"Avg unit cost: {avg_unit_cost:.2f} GBP")
print(f"Holding cost: {holding_cost:.2f} GBP/unit/year")
print()
print(f"EOQ: {eoq:,.0f} units per order")
print(f"Orders per year: {orders_per_year:.1f}")
print(f"Order cycle: {order_cycle:.1f} days")


ECONOMIC ORDER QUANTITY
Annual demand: 5,192,804 units
Ordering cost: 50 GBP/order
Avg unit cost: 1.65 GBP
Holding cost: 0.33 GBP/unit/year

EOQ: 39,638 units per order
Orders per year: 131.0
Order cycle: 2.8 days


## Inventory Simulation

In [7]:
# Simulate inventory levels over historical period
rop = reorder_points["95%"]
ss = safety_stocks["95%"]
order_qty = round(eoq)

inventory = []
current_stock = order_qty
pending_orders = []

for _, row in daily.iterrows():
    # Receive pending orders
    new_pending = []
    for order_day, qty in pending_orders:
        if (row["Date"] - order_day).days >= LEAD_TIME_DAYS:
            current_stock += qty
        else:
            new_pending.append((order_day, qty))
    pending_orders = new_pending

    # Fulfill demand
    demand = int(row["total_quantity"])
    current_stock -= demand
    stockout = max(0, -current_stock)
    current_stock = max(0, current_stock)

    # Reorder check
    ordered = 0
    if current_stock <= rop:
        pending_orders.append((row["Date"], order_qty))
        ordered = order_qty

    inventory.append({
        "Date": row["Date"],
        "demand": demand,
        "stock_level": current_stock,
        "stockout": stockout,
        "ordered": ordered,
    })

inv_df = pd.DataFrame(inventory)
stockout_days = (inv_df["stockout"] > 0).sum()
fill_rate = 1 - stockout_days / len(inv_df)

print(f"Simulation: {len(inv_df)} days")
print(f"Stockout days: {stockout_days} ({stockout_days/len(inv_df)*100:.1f}%)")
print(f"Fill rate: {fill_rate*100:.1f}%")
print(f"Total orders placed: {(inv_df['ordered'] > 0).sum()}")


Simulation: 739 days
Stockout days: 8 (1.1%)
Fill rate: 98.9%
Total orders placed: 268


In [8]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

axes[0,0].plot(inv_df["Date"], inv_df["stock_level"], color="#3498db", linewidth=1)
axes[0,0].axhline(rop, color="#e74c3c", linestyle="--", alpha=0.7, label=f"Reorder Point ({rop:,})")
axes[0,0].axhline(ss, color="#f39c12", linestyle="--", alpha=0.7, label=f"Safety Stock ({ss:,})")
axes[0,0].fill_between(inv_df["Date"], 0, ss, alpha=0.1, color="#f39c12")
axes[0,0].set_title("Inventory Levels Over Time"); axes[0,0].legend(fontsize=9)
axes[0,0].set_ylabel("Units")

axes[0,1].bar(inv_df["Date"], inv_df["demand"], color="#2ecc71", alpha=0.7, width=1)
axes[0,1].set_title("Daily Demand"); axes[0,1].set_ylabel("Units")

stockout_mask = inv_df["stockout"] > 0
axes[1,0].bar(inv_df.loc[stockout_mask, "Date"], inv_df.loc[stockout_mask, "stockout"],
              color="#e74c3c", alpha=0.8, width=1)
axes[1,0].set_title(f"Stockout Events ({stockout_days} days)"); axes[1,0].set_ylabel("Units Short")

order_mask = inv_df["ordered"] > 0
axes[1,1].bar(inv_df.loc[order_mask, "Date"], inv_df.loc[order_mask, "ordered"],
              color="#9b59b6", alpha=0.8, width=1)
axes[1,1].set_title(f"Reorders Placed ({order_mask.sum()} orders)"); axes[1,1].set_ylabel("Units")

fig.suptitle("Inventory Optimization Simulation", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "36_inventory_simulation.png")
plt.show()


Saved: 36_inventory_simulation.png


In [9]:
# Save inventory analysis
inv_summary = pd.DataFrame({
    "Metric": ["EOQ", "Safety Stock (95%)", "Reorder Point (95%)", "Lead Time",
               "Fill Rate", "Stockout Days", "Orders/Year", "Order Cycle"],
    "Value": [f"{eoq:,.0f} units", f"{ss:,} units", f"{rop:,} units",
              f"{LEAD_TIME_DAYS} days", f"{fill_rate*100:.1f}%",
              f"{stockout_days}", f"{orders_per_year:.1f}", f"{order_cycle:.1f} days"]
})
inv_summary.to_csv(os.path.join(PROCESSED_DIR, "inventory_metrics.csv"), index=False)
inv_df.to_csv(os.path.join(PROCESSED_DIR, "inventory_simulation.csv"), index=False)
print("Saved: inventory_metrics.csv, inventory_simulation.csv")
print()
print("INVENTORY OPTIMIZATION COMPLETE")


Saved: inventory_metrics.csv, inventory_simulation.csv

INVENTORY OPTIMIZATION COMPLETE
